# Projeto: Radix Sort Paralelo com Counting Sort Estável
Implementação limpa usando Python e Numba CUDA no Google Colab.

In [1]:
# 2. Instalação e verificação do ambiente
import sys
import os

def verificar_ambiente_cuda():
    print("=== Verificação do Ambiente de Execução ===")
    print(f"Versão do Python: {sys.version}")

    try:
        import numba
        from numba import cuda
        print(f"Versão do Numba: {numba.__version__}")
    except ImportError:
        print("Erro: Numba não está instalado. Por favor, execute: !pip install numba")
        return False

    gpus = cuda.gpus.lst
    if not gpus:
        print("\n[!] ALERTA: Nenhuma GPU CUDA foi detectada neste ambiente!")
        print("Como ativar a GPU no Google Colab:")
        print("1. No menu superior, clique em: Ambiente de execução -> Alterar tipo de ambiente de execução")
        print("2. Em 'Acelerador de hardware', selecione 'T4 GPU' (ou outra GPU disponível)")
        print("3. Clique em Salvar e execute esta célula novamente.")
        return False

    gpu = gpus[0]
    with gpu:
        # gpu.name já é do tipo string nas versões atuais, não requer .decode()
        gpu_nome_str = gpu.name if isinstance(gpu.name, str) else gpu.name.decode('utf-8')
        print(f"\nGPU Detectada: {gpu_nome_str}")
        # Verificação do compilador JIT
        try:
            @cuda.jit
            def dummy_kernel():
                pass
            dummy_kernel[1, 1]()
            cuda.synchronize()
            print("Compilador JIT CUDA do Numba: FUNCIONAL (Teste com sucesso!)")
            return True
        except Exception as e:
            print(f"Erro ao compilar kernel de teste JIT CUDA: {str(e)}")
            return False

_ = verificar_ambiente_cuda()

=== Verificação do Ambiente de Execução ===
Versão do Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Versão do Numba: 0.61.2

GPU Detectada: Tesla T4


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Compilador JIT CUDA do Numba: FUNCIONAL (Teste com sucesso!)


In [2]:
# 3. Importações e configuração global
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import io
from numba import cuda, int32, uint32, float32

# Parâmetros de Configuração Algorítmica
RADIX_BITS = 8
NUM_PASSES = 4
BUCKETS = 256 # 2^8
MASK = 255

# Tamanho do bloco padrão de processamento na GPU
THREADS_PER_BLOCK = 256

In [3]:
# 4. Leitura do CSV, geração de dados sintéticos e validação flexível
def ler_csv(caminho_ou_buffer):
    """
    Lê o arquivo CSV de entrada. Cada linha contém uma lista de inteiros separados por vírgula ou ponto e vírgula.
    """
    linhas = []
    if isinstance(caminho_ou_buffer, str):
        with open(caminho_ou_buffer, 'r') as f:
            conteudo_linhas = f.read().splitlines()
    else:
        conteudo_linhas = caminho_ou_buffer.read().splitlines()

    if not conteudo_linhas or len(conteudo_linhas) == 0:
        raise ValueError("Erro de Validação: O arquivo está vazio.")

    for i, linha in enumerate(conteudo_linhas):
        linha_limpa = linha.strip()
        if not linha_limpa:
            continue
        delimitador = ';' if ';' in linha_limpa else ','
        partes = [int(val) for val in linha_limpa.split(delimitador) if val.strip() != '']
        linhas.append(partes)

    if len(linhas) == 0:
        raise ValueError("Erro de Validação: Nenhuma linha válida pôde ser lida do CSV.")

    return validar_dados(linhas)

def validar_dados(linhas):
    N = len(linhas)

    # Valida individualmente os limites de cada elemento para int32
    for idx, linha in enumerate(linhas):
        for val in linha:
            if val < -2147483648 or val > 2147483647:
                raise ValueError(f"Erro de Validação: O valor {val} está fora do intervalo permitido para int32.")

    # Como as linhas possuem comprimentos variáveis, criamos uma lista de arrays numpy
    # e mapeamos dinamicamente para o processamento de forma robusta
    print(f"Dados carregados e validados com sucesso!")
    print(f"N (vetores/linhas): {N}.")
    for idx in range(min(5, N)):
        print(f"  - Vetor {idx} possui {len(linhas[idx])} elementos.")

    return linhas

def gerar_dados_teste(N, X, valor_min=-100000, valor_max=100000, seed=42):
    """
    Gera matriz contígua aleatória de teste para experimentos repetíveis.
    """
    rng = np.random.default_rng(seed)
    valor_min = max(valor_min, -2147483648)
    valor_max = min(valor_max, 2147483647)
    return rng.integers(valor_min, valor_max, size=(N, X), dtype=np.int32)

In [4]:
def ler_csv(caminho_ou_buffer):
    """
    Lê o arquivo CSV de entrada. Suporta delimitadores de vírgula (,) ou ponto e vírgula (;)
    e lida robustamente com diferentes quebras de linha (LF/CRLF).
    """
    linhas = []
    if isinstance(caminho_ou_buffer, str):
        with open(caminho_ou_buffer, 'r', encoding='utf-8', errors='ignore') as f:
            conteudo = f.read()
    else:
        conteudo = caminho_ou_buffer.read()
        if isinstance(conteudo, bytes):
            conteudo = conteudo.decode('utf-8', errors='ignore')

    # Substitui quebras do Windows por Linux e divide as linhas de forma limpa
    linhas_brutas = conteudo.replace('\r\n', '\n').replace('\r', '\n').split('\n')

    for i, linha in enumerate(linhas_brutas):
        linha_limpa = linha.strip()
        if not linha_limpa:
            continue

        # Identifica se o delimitador é ';' ou ','
        delimitador = ';' if ';' in linha_limpa else ','

        # Extrai os inteiros filtrando partes vazias
        partes = [int(val) for val in linha_limpa.split(delimitador) if val.strip() != '']
        if len(partes) > 0:
            linhas.append(partes)

    if len(linhas) == 0:
        raise ValueError("Erro de Validação: Nenhuma linha válida pôde ser lida do CSV.")

    # Imprime diagnóstico das primeiras linhas para auditoria
    print("=== Diagnóstico de Estrutura do CSV ===")
    for idx in range(min(5, len(linhas))):
        print(f"Linha {idx}: {len(linhas[idx])} elementos encontrados.")
    print("=======================================")

    return validar_dados(linhas)

In [5]:
# Executa o processamento com o diagnóstico detalhado ativo
caminho_entrada = "dados.csv"

try:
    # Lê, valida e ordena na GPU
    matriz_dados = ler_csv(caminho_entrada)
    matriz_ordenada_gpu = radix_sort_gpu(matriz_dados)

    print("\nPrévia da Matriz Ordenada na GPU (Primeiras 3 linhas):")
    print(matriz_ordenada_gpu[:3])

    # Salva o resultado ordenado
    salvar_resultado_csv(matriz_ordenada_gpu, "resultado_ordenado_gpu.csv")
except Exception as e:
    print(f"Ocorreu um erro durante o processamento: {str(e)}")

=== Diagnóstico de Estrutura do CSV ===
Linha 0: 1000000 elementos encontrados.
Linha 1: 2000000 elementos encontrados.
Linha 2: 3000000 elementos encontrados.
Linha 3: 1000000 elementos encontrados.
Linha 4: 2000000 elementos encontrados.
Dados carregados e validados com sucesso!
N (vetores/linhas): 27.
  - Vetor 0 possui 1000000 elementos.
  - Vetor 1 possui 2000000 elementos.
  - Vetor 2 possui 3000000 elementos.
  - Vetor 3 possui 1000000 elementos.
  - Vetor 4 possui 2000000 elementos.
Ocorreu um erro durante o processamento: name 'radix_sort_gpu' is not defined


## 5. Definição dos Kernels CUDA

In [6]:
# Kernels CUDA explícitos para o Radix Sort com Counting Sort Estável

@cuda.jit(device=True)
def obter_digito(valor, num_passagem):
    # Transformação de chave para ordenar inteiros de 32 bits com sinal de maneira correta
    chave = uint32(valor) ^ uint32(0x80000000)
    deslocamento = num_passagem * 8
    return (chave >> deslocamento) & uint32(255)

@cuda.jit
def calcular_histogramas_parciais_kernel(d_entrada, d_hist_parciais, num_passagem, N, X, elementos_por_bloco):
    """
    Cada bloco computa o histograma local de um fragmento do vetor de uma determinada linha.
    As partições das linhas são divididas fisicamente no grid da GPU.
    """
    linha = cuda.blockIdx.y
    bloco_x = cuda.blockIdx.x
    tid = cuda.threadIdx.x

    # Memória compartilhada para evitar colisões pesadas na memória global
    sh_hist = cuda.shared.array(shape=(256,), dtype=int32)

    # Inicializa memória compartilhada em paralelo por bloco
    if tid < 256:
        sh_hist[tid] = 0
    cuda.syncthreads()

    # Cada bloco cuida de um pedaço exclusivo (fragmento) do vetor
    inicio_idx = bloco_x * elementos_por_bloco
    fim_idx = min(inicio_idx + elementos_por_bloco, X)

    # Threads cooperam para ler os dados e calcular o histograma local com atomic_add
    idx = inicio_idx + tid
    while idx < fim_idx:
        if idx < X:
            val = d_entrada[linha, idx]
            digito = obter_digito(val, num_passagem)
            cuda.atomic.add(sh_hist, digito, 1)
        idx += cuda.blockDim.x

    cuda.syncthreads()

    # Escrita do histograma parciais gerado pelo bloco na matriz global
    if tid < 256:
        # Formato de indexação: [linha do vetor, id_bloco_fragmento, bucket_id]
        d_hist_parciais[linha, bloco_x, tid] = sh_hist[tid]

In [7]:
@cuda.jit
def reduzir_histogramas_kernel(d_hist_parciais, d_hist_global, N, num_blocos_fragmento):
    """
    Soma os histogramas parciais de cada vetor independente para gerar um histograma global por linha.
    """
    linha = cuda.blockIdx.x
    tid = cuda.threadIdx.x # Cada thread cuida de um dos 256 buckets

    if linha < N and tid < 256:
        soma = 0
        for f in range(num_blocos_fragmento):
            soma += d_hist_parciais[linha, f, tid]
        d_hist_global[linha, tid] = soma

In [8]:
@cuda.jit
def calcular_prefix_sum_kernel(d_hist_global, d_prefix_sum, N):
    """
    Executa um prefix-sum paralelo (Exclusivo) nos histogramas globais de cada vetor independente.
    Como a largura é sempre de 256 caminhos, o cálculo paralelo é executado em um único bloco por vetor.
    """
    linha = cuda.blockIdx.x
    tid = cuda.threadIdx.x

    # Aloca memória compartilhada local do bloco
    temp = cuda.shared.array(shape=(256,), dtype=int32)

    if linha < N and tid < 256:
        temp[tid] = d_hist_global[linha, tid]
    else:
        temp[tid] = 0

    cuda.syncthreads()

    # Varredura de prefix-sum paralelo Hillis-Steele simplificado para 256 elementos
    deslocamento = 1
    while deslocamento < 256:
        val = 0
        if tid >= deslocamento:
            val = temp[tid - deslocamento]
        cuda.syncthreads()
        temp[tid] += val
        cuda.syncthreads()
        deslocamento *= 2

    # Converte o prefix-sum de inclusivo para exclusivo e salva na memória global
    if linha < N and tid < 256:
        if tid == 0:
            d_prefix_sum[linha, 0] = 0
        else:
            d_prefix_sum[linha, tid] = temp[tid - 1]

In [9]:
@cuda.jit
def calcular_offsets_dos_blocos_kernel(d_hist_parciais, d_offsets_blocos, N, num_blocos_fragmento):
    """
    Calcula o prefix sum ao longo do eixo dos fragmentos (blocos_x) para cada bucket do histograma.
    Isto determina os offsets iniciais de escrita de cada fragmento em sua respectiva linha.
    """
    linha = cuda.blockIdx.x
    bucket = cuda.threadIdx.x # Uma thread por bucket (256 buckets)

    if linha < N and bucket < 256:
        soma_acumulada = 0
        for f in range(num_blocos_fragmento):
            d_offsets_blocos[linha, f, bucket] = soma_acumulada
            soma_acumulada += d_hist_parciais[linha, f, bucket]

In [10]:
@cuda.jit
def distribuir_elementos_estavelmente_kernel(d_entrada, d_saida, d_prefix_sum, d_offsets_blocos, num_passagem, N, X, elementos_por_bloco):
    """
    Escreve de forma paralela e estável cada elemento no seu buffer de destino correspondente.
    """
    linha = cuda.blockIdx.y
    bloco_x = cuda.blockIdx.x
    tid = cuda.threadIdx.x

    inicio_idx = bloco_x * elementos_por_bloco
    fim_idx = min(inicio_idx + elementos_por_bloco, X)

    # Memória local para rastrear as posições sequenciais locais dentro do fragmento do bloco
    pos_local = cuda.shared.array(shape=(256,), dtype=int32)

    # Inicializa memória compartilhada
    if tid < 256:
        pos_local[tid] = 0
    cuda.syncthreads()

    # Para manter estabilidade pura sem contenção de concorrência ativa,
    # processamos serialmente o pequeno fragmento local da thread ou cooperamos localmente
    idx = inicio_idx + tid
    if idx < fim_idx:
        val = d_entrada[linha, idx]
        digito = obter_digito(val, num_passagem)

        # Contagem local de elementos à esquerda de mim dentro deste mesmo bloco com mesma chave
        offset_interno = 0
        for j in range(inicio_idx, idx):
            v_j = d_entrada[linha, j]
            digito_j = obter_digito(v_j, num_passagem)
            if digito_j == digito:
                offset_interno += 1

        # Destino final baseado no prefix-sum global, offset de blocos e indexação interna
        posicao_final = d_prefix_sum[linha, digito] + d_offsets_blocos[linha, bloco_x, digito] + offset_interno
        d_saida[linha, posicao_final] = val

In [11]:
# 6. Implementação do Radix Sort Orquestrador na GPU (compatível com linhas de tamanhos dinâmicos)
def radix_sort_gpu(linhas_entrada):
    """
    Orquestra as passagens do Radix Sort na GPU suportando linhas com tamanhos de vetor independentes.
    """
    # Se recebemos uma lista de linhas de comprimentos variáveis
    if isinstance(linhas_entrada, list):
        resultados = []
        for i, linha in enumerate(linhas_entrada):
            array_linha = np.array(linha, dtype=np.int32)
            resultado_linha = _ordenar_vetor_unico_gpu(array_linha)
            resultados.append(resultado_linha)
        # Retorna no mesmo formato estrutural flexível para salvar
        return resultados
    else:
        # Caso seja uma matriz contígua padrão numpy
        N, X = linhas_entrada.shape
        resultados = np.empty_like(linhas_entrada)
        for i in range(N):
            resultados[i] = _ordenar_vetor_unico_gpu(linhas_entrada[i])
        return resultados

def _ordenar_vetor_unico_gpu(vetor_1d):
    """
    Auxiliar que ordena de forma paralela e isolada um único vetor de tamanho genérico X na GPU.
    """
    X = len(vetor_1d)
    N = 1 # Processamento de vetor individual

    elementos_por_bloco = THREADS_PER_BLOCK
    num_blocos_fragmento = (X + elementos_por_bloco - 1) // elementos_por_bloco
    if num_blocos_fragmento == 0:
        num_blocos_fragmento = 1

    # Converte para formato 2D para ser consumido perfeitamente pelos Kernels CUDA existentes
    matriz_2d = np.ascontiguousarray(vetor_1d.reshape(1, X))

    d_buffer_A = cuda.to_device(matriz_2d)
    d_buffer_B = cuda.device_array_like(matriz_2d)

    d_hist_parciais = cuda.device_array((1, num_blocos_fragmento, 256), dtype=np.int32)
    d_hist_global = cuda.device_array((1, 256), dtype=np.int32)
    d_prefix_sum = cuda.device_array((1, 256), dtype=np.int32)
    d_offsets_blocos = cuda.device_array((1, num_blocos_fragmento, 256), dtype=np.int32)

    for passagem in range(NUM_PASSES):
        grid_hist = (num_blocos_fragmento, 1)

        calcular_histogramas_parciais_kernel[grid_hist, THREADS_PER_BLOCK](
            d_buffer_A, d_hist_parciais, passagem, 1, X, elementos_por_bloco
        )

        reduzir_histogramas_kernel[1, 256](
            d_hist_parciais, d_hist_global, 1, num_blocos_fragmento
        )

        calcular_prefix_sum_kernel[1, 256](
            d_hist_global, d_prefix_sum, 1
        )

        calcular_offsets_dos_blocos_kernel[1, 256](
            d_hist_parciais, d_offsets_blocos, 1, num_blocos_fragmento
        )

        distribuir_elementos_estavelmente_kernel[grid_hist, THREADS_PER_BLOCK](
            d_buffer_A, d_buffer_B, d_prefix_sum, d_offsets_blocos, passagem, 1, X, elementos_por_bloco
        )

        d_buffer_A, d_buffer_B = d_buffer_B, d_buffer_A

    cuda.synchronize()
    return d_buffer_A.copy_to_host().flatten()

In [12]:
# 7. Referência sequencial em Python
def counting_sort_estavel_seq(vetor, passagem):
    N_elem = len(vetor)
    saida = np.zeros(N_elem, dtype=np.int32)
    contagem = np.zeros(256, dtype=np.int32)

    # Extração de chaves e contagem
    for val in vetor:
        # Conversão de sinal para ordenação consistente de inteiros de 32 bits
        chave = uint32(val) ^ uint32(0x80000000)
        deslocamento = passagem * 8
        digito = (chave >> deslocamento) & 255
        contagem[digito] += 1

    # Somas acumuladas
    for i in range(1, 256):
        contagem[i] += contagem[i-1]

    # Distribuição reversa para estabilidade do Counting Sort
    for val in reversed(vetor):
        chave = uint32(val) ^ uint32(0x80000000)
        deslocamento = passagem * 8
        digito = (chave >> deslocamento) & 255
        posicao = contagem[digito] - 1
        saida[posicao] = val
        contagem[digito] -= 1

    return saida

def radix_sort_sequencial(matriz):
    N, X = matriz.shape
    matriz_saida = np.empty_like(matriz)
    for i in range(N):
        vetor_at = matriz[i].copy()
        for passagem in range(NUM_PASSES):
            vetor_at = counting_sort_estavel_seq(vetor_at, passagem)
        matriz_saida[i] = vetor_at
    return matriz_saida

In [13]:
# 8. Testes de correção automática
def validar_resultado(original, ordenado):
    if original.shape != ordenado.shape:
        return False, "Dimensões diferentes da matriz original."

    for i in range(original.shape[0]):
        lin_orig = original[i]
        lin_ord = ordenado[i]

        # 1. Verifica se está ordenado de forma não decrescente
        if not np.all(lin_ord[:-1] <= lin_ord[1:]):
            return False, f"Vetor na linha {i} não está ordenado corretamente."

        # 2. Verifica conservação de elementos
        if not np.array_equal(np.sort(lin_orig), lin_ord):
            return False, f"Elementos corrompidos ou modificados na linha {i}."

    return True, "Todos os testes de ordenação paralela passaram com sucesso!"

def testar_correcao():
    print("Iniciando Testes de Correção...")

    casos_teste = {
        "Vetor Único Elemento": gerar_dados_teste(1, 1),
        "Vetor Já Ordenado": np.array([[1, 2, 3, 4, 5, 10, 50, 100]], dtype=np.int32),
        "Vetor Decrescente": np.array([[100, 99, 50, 20, 10, 5, 1, 0, -5]], dtype=np.int32),
        "Valores Repetidos": np.array([[10, 10, -5, -5, 0, 0, 20, 20]], dtype=np.int32),
        "Valores Próximos de Limites int32": np.array([[-2147483648, 2147483647, 0, -1, 1]], dtype=np.int32),
        "Matriz Multilinha": gerar_dados_teste(5, 513) # Tamanho não múltiplo do bloco
    }

    for nome, dados in casos_teste.items():
        resultado_gpu = radix_sort_gpu(dados)
        passou, erro_msg = validar_resultado(dados, resultado_gpu)
        if passou:
            print(f"[PASSOU] - {nome}")
        else:
            print(f"[FALHOU] - {nome}: {erro_msg}")

testar_correcao()

Iniciando Testes de Correção...


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPe

[PASSOU] - Vetor Único Elemento
[PASSOU] - Vetor Já Ordenado
[PASSOU] - Vetor Decrescente
[PASSOU] - Valores Repetidos
[PASSOU] - Valores Próximos de Limites int32
[PASSOU] - Matriz Multilinha


In [14]:
# 11. Exportação do CSV ordenado adaptado para comprimentos variáveis
def salvar_resultado_csv(matriz_ou_lista_ordenada, caminho_arquivo="vetores_ordenados.csv"):
    with open(caminho_arquivo, 'w', encoding='utf-8') as f:
        for linha in matriz_ou_lista_ordenada:
            linha_str = ";".join(map(str, linha))
            f.write(linha_str + "\n")
    print(f"Arquivo '{caminho_arquivo}' salvo com sucesso!")

In [15]:
# Carrega o arquivo dados.csv que você colocou na pasta
caminho_entrada = "dados.csv"

try:
    # Lê e valida os dados
    matriz_dados = ler_csv(caminho_entrada)

    # Executa a ordenação paralela na GPU
    matriz_ordenada_gpu = radix_sort_gpu(matriz_dados)

    # Exibe uma prévia dos resultados
    print("\nPrévia da Matriz Ordenada na GPU (Primeiras 3 linhas):")
    print(matriz_ordenada_gpu[:3])

    # Salva o arquivo de saída
    salvar_resultado_csv(matriz_ordenada_gpu, "resultado_ordenado_gpu.csv")
except FileNotFoundError:
    print(f"Erro: O arquivo '{caminho_entrada}' não foi encontrado na pasta atual do Colab. Certifique-se de que o upload foi concluído com este nome exato.")
except Exception as e:
    print(f"Ocorreu um erro durante o processamento: {str(e)}")

=== Diagnóstico de Estrutura do CSV ===
Linha 0: 1000000 elementos encontrados.
Linha 1: 2000000 elementos encontrados.
Linha 2: 3000000 elementos encontrados.
Linha 3: 1000000 elementos encontrados.
Linha 4: 2000000 elementos encontrados.
Dados carregados e validados com sucesso!
N (vetores/linhas): 27.
  - Vetor 0 possui 1000000 elementos.
  - Vetor 1 possui 2000000 elementos.
  - Vetor 2 possui 3000000 elementos.
  - Vetor 3 possui 1000000 elementos.
  - Vetor 4 possui 2000000 elementos.

Prévia da Matriz Ordenada na GPU (Primeiras 3 linhas):
[array([       191,       5234,       8066, ..., 2147477468, 2147478709,
       2147482794], dtype=int32), array([      1814,       2244,       6800, ..., 2147481131, 2147481628,
       2147482790], dtype=int32), array([      1063,       1359,       2452, ..., 2147482474, 2147483350,
       2147483567], dtype=int32)]
Arquivo 'resultado_ordenado_gpu.csv' salvo com sucesso!
